# LLM-Based GORE Pipeline — Execution

This notebook executes the complete Goal-Oriented Requirements Engineering pipeline. It separates the baseline top-down architecture from the proposed iterative bottom-up extension and keeps metric computation outside the execution workflow.

The execution flow is:

1. environment and dataset configuration;
2. baseline extraction and refinement of actors, high-level goals, and low-level goals;
3. bounded refinement–abstraction–verification cycle between high-level and low-level goals;
4. persistence of all intermediate and final artifacts;
5. goal-to-API alignment using the final verified low-level goals.

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from key import (
    get_key_openai,
    get_key_llama,
    count_Llama_keys,
)

openai_key = get_key_openai()
groq_key = get_key_llama()

print("Chiave OpenAI presente:", bool(openai_key))
print("Formato OpenAI plausibile:", openai_key.startswith("sk-"))

print("Chiave Groq presente:", bool(groq_key))
print("Formato Groq plausibile:", groq_key.startswith("gsk_"))

print("Numero chiavi Groq:", count_Llama_keys())

Project root: c:\Users\agnes_lryeu3v\Desktop\tesi
Chiave OpenAI presente: True
Formato OpenAI plausibile: True
Chiave Groq presente: True
Formato Groq plausibile: True
Numero chiavi Groq: 5


## 1. Environment and configuration

This section configures the project path, imports the pipeline components, selects the prompting mode, enables or disables the LLaMA ablation, and loads the datasets used during execution.


In [ ]:
from pathlib import Path
from threading import Thread
import json
import os
import sys

# Resolve the project root whether the notebook is launched from the project
# directory or from a notebooks/ subdirectory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from groundtruth import (
    GENOME,
    GESTAO_HOSPITAL,
    SIA_PROJECT_25_26,
    LONDON_AMBULANCE_SYSTEM,
)
from src.extraction.extractor import (
    generate_description,
    generate_actors,
    generate_high_level_goals,
    generate_low_level_goals,
)
from src.mapping.APIs_mapping import (
    generate_mapping_apis_goals,
    print_api_goal_mapping,
)
from src.self_critique.refine_response import (
    EvalMode,
    generate_response_with_reflection,
)
from src.utils import get_api_list_from_swagger
from src.examples.shot_learning import ShotPromptingMode

GROUNDTRUTHS = [
    GENOME,
    GESTAO_HOSPITAL,
    LONDON_AMBULANCE_SYSTEM,
    SIA_PROJECT_25_26,
]

LLAMA_ABLATION = True
PROMPTING_MODE = ShotPromptingMode.FEW_SHOT
OUTPUT_PATH = PROJECT_ROOT / "output"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)


def output_file_for(dataset_name: str) -> Path:
    suffix = "_noLlama" if LLAMA_ABLATION else ""
    return OUTPUT_PATH / f"{dataset_name}_{PROMPTING_MODE.name}{suffix}.json"


## 2. Baseline top-down pipeline

The baseline architecture follows a top-down process. Starting from the textual documentation, it extracts actors, identifies high-level goals associated with those actors, and decomposes each high-level goal into operational low-level goals. Each extraction stage is executed through the existing reflection-based refinement mechanism.

The original architecture is kept conceptually separate from the proposed extension so that its outputs can be reused as a baseline during the experimental evaluation.


### 2.1 Actor extraction

The actor extractor identifies the entities that interact with the system or pursue goals through it. The reflection loop evaluates and, when required, refines the generated actor set.

### 2.2 High-level goal extraction

The high-level goal extractor derives abstract stakeholder or system intentions from the documentation and associates them with the previously extracted actors.

### 2.3 Low-level goal extraction

The low-level goal extractor decomposes the high-level goals into concrete and operational objectives. These definitive low-level goals become the input of the proposed bottom-up reconstruction stage.


In [ ]:
generated_descriptions = {}
generated_actors = {}
generated_hl = {}
generated_ll = {}
generated_actor_objects = {}
generated_hl_objects = {}
generated_ll_objects = {}
baseline_errors = {}


def run_baseline_for_dataset(groundtruth: dict) -> None:
    dataset_name = groundtruth["name"]

    try:
        description = (
            str(generate_description(groundtruth["link-readme"]))
            if "link-readme" in groundtruth
            else groundtruth["description"]
        )
        generated_descriptions[dataset_name] = description

        actors, actors_score, actors_critique = generate_response_with_reflection(
            "Actors",
            generate_actors,
            define_args=(description,),
            eval_mode=EvalMode.ACTORS,
            eval_args=(description,),
            shotPromptingMode=PROMPTING_MODE,
            llama_ablation=LLAMA_ABLATION,
        )

        generated_actor_objects[dataset_name] = actors
        generated_actors[dataset_name] = [actor.name for actor in actors.actors]

        high_level_goals, hl_score, hl_critique = generate_response_with_reflection(
            "High Level Goals",
            generate_high_level_goals,
            define_args=(description, actors),
            eval_mode=EvalMode.HIGH_LEVEL,
            eval_args=(description, actors),
            shotPromptingMode=PROMPTING_MODE,
            llama_ablation=LLAMA_ABLATION,
        )

        generated_hl_objects[dataset_name] = high_level_goals
        generated_hl[dataset_name] = [
            goal.description for goal in high_level_goals.goals
        ]

        low_level_goals, ll_score, ll_critique = generate_response_with_reflection(
            "Low Level Goals",
            generate_low_level_goals,
            define_args=(high_level_goals,),
            eval_mode=EvalMode.LOW_LEVEL,
            eval_args=(description, actors, high_level_goals),
            shotPromptingMode=PROMPTING_MODE,
            llama_ablation=LLAMA_ABLATION,
        )

        generated_ll_objects[dataset_name] = low_level_goals
        generated_ll[dataset_name] = [
            goal.description for goal in low_level_goals.low_level_goals
        ]

        output = {
            "name": dataset_name,
            "description": description,
            "actors": generated_actors[dataset_name],
            "highLevelGoals": generated_hl[dataset_name],
            "lowLevelGoals": generated_ll[dataset_name],
            "baselineScores": {
                "actors": actors_score,
                "highLevelGoals": hl_score,
                "lowLevelGoals": ll_score,
            },
            "baselineCritiques": {
                "actors": actors_critique,
                "highLevelGoals": hl_critique,
                "lowLevelGoals": ll_critique,
            },
        }

        with output_file_for(dataset_name).open("w", encoding="utf-8") as file:
            json.dump(output, file, indent=4, ensure_ascii=False, default=str)

        print(f"[{dataset_name}] Baseline extraction completed.")

    except Exception as error:
        baseline_errors[dataset_name] = f"{type(error).__name__}: {error}"
        print(f"[{dataset_name}] Baseline extraction failed: {error}")


threads = []
for groundtruth in GROUNDTRUTHS:
    thread = Thread(target=run_baseline_for_dataset, args=(groundtruth,))
    thread.start()
    threads.append(thread)

for thread in threads:
    thread.join()

print(f"Completed datasets: {len(generated_ll_objects)} / {len(GROUNDTRUTHS)}")
if baseline_errors:
    print("Baseline errors:", baseline_errors)


## 3. Proposed iterative bottom-up extension

The proposed extension starts after the baseline low-level goals have been produced. Instead of executing bottom-up reconstruction, global evaluation, and low-level regeneration only once, the notebook delegates the complete bounded feedback loop to `run_global_goal_cycle`.

For each iteration, the orchestrator:

1. reconstructs one candidate high-level goal for every non-empty branch from its low-level goals;
2. globally evaluates each reconstructed candidate against its parent, the complete high-level goal collection, and the project description;
3. checks whether every expected branch is confirmed;
4. when convergence is not reached, deterministically updates the high-level goal collection and regenerates the complete low-level decomposition;
5. repeats until all branches are confirmed or a stopping condition is reached.

The orchestration logic remains deterministic. LLMs are used only inside the existing reconstruction, evaluation, and generation components.

### 3.1 Complete refinement–abstraction–verification cycle

The cycle receives the definitive baseline `HighLevelGoals` and `LowLevelGoals` objects. The existing reflection-based low-level generator is injected into the orchestrator as a callback, so no new low-level generation mechanism is introduced.

Each iteration is persisted through `GlobalGoalCycleIteration`, including input goals, branch traceability, bottom-up reconstructions, evaluator decisions, errors, updated high-level goals, regenerated low-level goals, convergence status, and the deterministic state signature.

In [ ]:
from src.bottom_up.goal_cycle_orchestrator import run_global_goal_cycle

MAX_GLOBAL_CYCLE_ITERATIONS = 3

global_cycle_results = {}
global_cycle_errors = {}
final_high_level_goals_objects = {}
final_low_level_goals_objects = {}

for dataset_name, initial_high_level_goals in generated_hl_objects.items():
    if dataset_name not in generated_ll_objects:
        global_cycle_errors[dataset_name] = (
            "Initial low-level goals are not available."
        )
        print(
            f"[{dataset_name}] Global goal cycle skipped: "
            "initial low-level goals are not available."
        )
        continue

    initial_low_level_goals = generated_ll_objects[dataset_name]
    description = generated_descriptions[dataset_name]
    actors = generated_actor_objects[dataset_name]

    def regenerate_low_level_goals_for_cycle(
        high_level_goals,
        _description=description,
        _actors=actors,
    ):
        regenerated, _score, _critique = generate_response_with_reflection(
            "Low Level Goals",
            generate_low_level_goals,
            define_args=(high_level_goals,),
            eval_mode=EvalMode.LOW_LEVEL,
            eval_args=(
                _description,
                _actors,
                high_level_goals,
            ),
            shotPromptingMode=PROMPTING_MODE,
            llama_ablation=LLAMA_ABLATION,
        )
        return regenerated

    try:
        cycle_result = run_global_goal_cycle(
            project_description=description,
            initial_high_level_goals=initial_high_level_goals,
            initial_low_level_goals=initial_low_level_goals,
            regenerate_low_level_goals=(
                regenerate_low_level_goals_for_cycle
            ),
            max_iterations=MAX_GLOBAL_CYCLE_ITERATIONS,
        )
    except Exception as error:
        global_cycle_errors[dataset_name] = (
            f"{type(error).__name__}: {error}"
        )
        print(
            f"[{dataset_name}] Global goal cycle failed: "
            f"{type(error).__name__}: {error}"
        )
        continue

    global_cycle_results[dataset_name] = cycle_result
    final_high_level_goals_objects[dataset_name] = (
        cycle_result.final_high_level_goals
    )
    final_low_level_goals_objects[dataset_name] = (
        cycle_result.final_low_level_goals
    )

    output_file = output_file_for(dataset_name)
    with output_file.open("r", encoding="utf-8") as file:
        output = json.load(file)

    output["globalGoalCycle"] = cycle_result.model_dump(
        mode="json"
    )
    output["finalHighLevelGoals"] = (
        cycle_result.final_high_level_goals.model_dump(
            mode="json"
        )
    )
    output["finalLowLevelGoals"] = (
        cycle_result.final_low_level_goals.model_dump(
            mode="json"
        )
    )

    with output_file.open("w", encoding="utf-8") as file:
        json.dump(
            output,
            file,
            indent=4,
            ensure_ascii=False,
        )

    print(
        f"[{dataset_name}] Global cycle completed: "
        f"converged={cycle_result.converged}, "
        f"stop_reason={cycle_result.stop_reason}, "
        f"iterations={cycle_result.completed_iterations}."
    )

### 3.2 Cycle result inspection

The following cell prints a compact per-dataset summary. Detailed iteration artifacts are already persisted under `globalGoalCycle` in the corresponding JSON output.

In [ ]:
for dataset_name, cycle_result in global_cycle_results.items():
    print()
    print(f"[{dataset_name}]")
    print(f"  Converged: {cycle_result.converged}")
    print(f"  Stop reason: {cycle_result.stop_reason}")
    print(
        f"  Completed iterations: "
        f"{cycle_result.completed_iterations}"
    )
    print(
        f"  Final high-level goals: "
        f"{len(cycle_result.final_high_level_goals.goals)}"
    )
    print(
        f"  Final low-level goals: "
        f"{len(cycle_result.final_low_level_goals.low_level_goals)}"
    )

    if cycle_result.unresolved_bottom_up_errors:
        print(
            "  Unresolved bottom-up errors:",
            cycle_result.unresolved_bottom_up_errors,
        )

    if cycle_result.unresolved_global_evaluation_errors:
        print(
            "  Unresolved global-evaluation errors:",
            cycle_result.unresolved_global_evaluation_errors,
        )

    if cycle_result.unresolved_empty_branches:
        print(
            "  Unresolved empty branches:",
            cycle_result.unresolved_empty_branches,
        )

if global_cycle_errors:
    print()
    print("Global cycle execution errors:", global_cycle_errors)

### 3.3 Final goal collections

The final high-level and low-level goal collections are taken directly from `GlobalGoalCycleResult`. When the cycle converges, they correspond to the confirmed state. When a bounded or technical stopping condition is reached, they correspond to the last fully evaluated state, together with the recorded stop reason and unresolved errors.

In [ ]:
final_high_level_goals = {
    dataset_name: [
        goal.description
        for goal in goals.goals
    ]
    for dataset_name, goals in final_high_level_goals_objects.items()
}

final_low_level_goals = {
    dataset_name: [
        goal.description
        for goal in goals.low_level_goals
    ]
    for dataset_name, goals in final_low_level_goals_objects.items()
}

for dataset_name in final_low_level_goals_objects:
    print(
        f"[{dataset_name}] Final goal collections available: "
        f"{len(final_high_level_goals_objects[dataset_name].goals)} HLG, "
        f"{len(final_low_level_goals_objects[dataset_name].low_level_goals)} LLG."
    )

## 4. Output persistence and traceability

Each dataset JSON contains the baseline artifacts and the complete iterative-extension result:

- `actors`;
- `highLevelGoals`;
- `lowLevelGoals`;
- `baselineScores`;
- `baselineCritiques`;
- `globalGoalCycle`;
- `finalHighLevelGoals`;
- `finalLowLevelGoals`.

`globalGoalCycle` contains the convergence status, stop reason, final collections, unresolved errors, and the complete ordered trace of all iterations. The traceability metadata is persisted for analysis but is never exposed to the bottom-up generator.

In [ ]:
for groundtruth in GROUNDTRUTHS:
    dataset_name = groundtruth["name"]
    output_file = output_file_for(dataset_name)

    if not output_file.exists():
        print(f"[{dataset_name}] Output file not available.")
        continue

    with output_file.open("r", encoding="utf-8") as file:
        output = json.load(file)

    print(
        f"[{dataset_name}] Saved sections: "
        f"{', '.join(output.keys())}"
    )


## 5. Goal-to-API alignment

The API alignment stage remains part of the original architecture. It is executed after the iterative extension and consumes `finalLowLevelGoals`, namely the last fully evaluated low-level decomposition returned by the cycle.

### 5.1 API extraction from Swagger

For each dataset that provides a Swagger source, this block extracts the available APIs.


In [ ]:
api_lists = {}

for groundtruth in GROUNDTRUTHS:
    dataset_name = groundtruth["name"]

    if "swagger" not in groundtruth:
        print(f"[{dataset_name}] No Swagger source configured; skipping API extraction.")
        continue

    print(f"[{dataset_name}] API extraction started...")
    api_lists[dataset_name] = get_api_list_from_swagger(
        link=groundtruth["swagger"]
    )
    print(
        f"[{dataset_name}] API extraction completed: "
        f"{len(api_lists[dataset_name])} API(s)."
    )


### 5.2 API mapping to final low-level goals

Each extracted API list is mapped to the final low-level goals returned by the global cycle. Datasets for which the cycle could not produce a final result are skipped and recorded separately.

In [ ]:
api_mappings = {}
api_mapping_skipped_due_to_cycle_errors = {}

for dataset_name, api_list in api_lists.items():
    if dataset_name not in final_low_level_goals_objects:
        error = global_cycle_errors.get(
            dataset_name,
            "Final low-level goals are not available.",
        )
        api_mapping_skipped_due_to_cycle_errors[dataset_name] = error
        print(f"[{dataset_name}] API mapping skipped: {error}")
        continue

    low_level_goals_for_mapping = (
        final_low_level_goals_objects[dataset_name]
    )

    print(f"[{dataset_name}] API mapping started...")
    mappings = generate_mapping_apis_goals(
        low_level_goals_for_mapping,
        api_list,
    )
    api_mappings[dataset_name] = mappings

    print_api_goal_mapping(mappings)

    mapping_file = OUTPUT_PATH / (
        f"final_mapping_{dataset_name}_{PROMPTING_MODE.name}"
        f"{'_noLlama' if LLAMA_ABLATION else ''}.json"
    )

    with mapping_file.open("w", encoding="utf-8") as file:
        json.dump(
            [mapping.model_dump(mode="json") for mapping in mappings],
            file,
            indent=4,
            ensure_ascii=False,
        )

    print(f"[{dataset_name}] API mapping saved to {mapping_file}.")

## 6. Execution summary

This notebook intentionally does not compute precision, recall, F1, semantic-similarity curves, or LLM-as-a-judge comparison statistics. Those analyses should read the persisted JSON files from a separate experimental-evaluation notebook, ensuring that generation and evaluation remain reproducible and independently repeatable.


In [ ]:
print("Execution summary")
print("-----------------")
print(f"Configured datasets: {len(GROUNDTRUTHS)}")
print(f"Baseline completed: {len(generated_ll_objects)}")
print(f"Global cycles completed: {len(global_cycle_results)}")
print(
    "Converged global cycles: "
    f"{sum(1 for result in global_cycle_results.values() if result.converged)}"
)
print(
    "Cycles stopped without convergence: "
    f"{sum(1 for result in global_cycle_results.values() if not result.converged)}"
)
print(f"Final goal collections available: {len(final_low_level_goals_objects)}")
print(f"API mappings completed: {len(api_mappings)}")
print(f"Global cycle execution errors: {len(global_cycle_errors)}")
print(
    "API mappings skipped due to cycle errors: "
    f"{len(api_mapping_skipped_due_to_cycle_errors)}"
)

if baseline_errors:
    print("Baseline errors:", baseline_errors)

if global_cycle_errors:
    print("Global cycle errors:", global_cycle_errors)